In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import torch
import sys
import os
from torch.utils.data import DataLoader, random_split
import wandb
import random
import datetime
import time

from tqdm.auto import tqdm
import sys, time


current_dir = os.path.dirname(os.path.abspath(''))
two_up_dir = os.path.dirname(os.path.dirname(current_dir))
sys.path.append(two_up_dir)

from TinyCenterSpeed.src.models.resnet import *
from TinyCenterSpeed.src.models.CenterSpeed import *
from TinyCenterSpeed.dataset.CenterSpeed_dataset import *
from TinyCenterSpeed.src.models.losses import *
from train import *

# W&B
%env "WANDB_NOTEBOOK_NAME" "centerspeed_with_val.ipynb"
print("wandb version:", wandb.__version__)
wandb.login()

# (유지) 하이퍼파라미터/런 설정 - 원본 형식과 유사하게
use_wandb = True
save_code = True

epochs = 500

learning_rate = 5e-4
architecture = "CenterSpeed: Hourglass Deep with Sigmoid, & Dropout, BatchNorm and Head with 2 frames, lower resolution: 64x64, pixelsize 0.1"
dataset = "Transfer learning test"
optimizer_name = "Adam"
batch_size = 32
timer = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_name = "CenterSpeed_redbull" + timer
loss_used = "CenterSpeedLossFreev2 with updated logic for free tracks on the dataset level"

config = {
    "epochs": epochs,
    "learning_rate": learning_rate,
    "architecture": architecture,
    "dataset": dataset,
    "optimizer": optimizer_name,
    "Loss-Function": loss_used
}

if use_wandb:
    run = wandb.init(project="CenterSpeedLowRes", config=config, name=run_name, save_code=save_code)
    wandb.define_metric("epoch")
    wandb.define_metric("train/*", step_metric="epoch")
    wandb.define_metric("val/*", step_metric="epoch")
    wandb.define_metric("time/*", step_metric="epoch")


env: "WANDB_NOTEBOOK_NAME"="centerspeed_with_val.ipynb"
wandb version: 0.21.1


wandb: Currently logged in as: whdaudpark (whdaudpark-dongguk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
# Dataset (사용자 경로 그대로 예시; 필요시 수정)
data_root = '/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT'

# CenterSpeed_dataset.py 코드에서 정의된 CenterSpeedDataset 클래스를 사용하여 데이터셋을 생성
set = CenterSpeedDataset(data_root, transform=True, dense=True)

set.seq_len = 2
# x축과 y축 방향의 가우시안 표준편차(σ) 설정, 값이 클수록 넓게 퍼짐 
# 객체 주변 0.9 픽셀로 점 찍음 
# 2 ~ 5 정도로 하자 
set.sx = 2
set.sy = 2
set.change_image_size(128)
set.change_pixel_size(0.1)

# 요청하신 분할 비율로 변경
train_size = int(len(set) * 0.8)
val_size   = int(len(set) * 0.2)
test_size  = len(set) - (train_size + val_size)

train_dataset, val_dataset, test_dataset = random_split(set, [train_size, val_size, test_size])

print("Size of Training Set: ", len(train_dataset))
print("Size of Validation Set: ", len(val_dataset))
print("Size of Testing Set: ", len(test_dataset))

# DataLoaders (원본 스타일 유지)
training_loader   = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
testing_loader    = DataLoader(test_dataset, batch_size=32, shuffle=False)


Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/TrainGT_redbull_obs1_0812_no_intensity_more.csv
Entries     :  4190
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/TrainGT_redbull_obs1_0812_no_intensity.csv
Entries     :  11217
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/TrainGT_redbull_obs2_0812_no_intensity_more.csv
Entries     :  3951
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/TrainGT_redbull_obs2_0812_no_intensity.csv
Entries     :  11970
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/redbull_testobs1.csv
Entries     :  7330
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/TrainGT_redbull_obs3_0812_no_intensity_more.csv
Entries     :  3931
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/TrainGT_redbull_obs3_0812_no_intensity.csv
Entries     :  11140
Total rows :  53729
File index :  [(0, 4190), (4190, 15407), (15407, 19358), (19358, 31328), (31328, 38658), (38658, 42589

In [3]:
# Model / Optimizer / Loss (원본 형식 그대로 유지)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

net = CenterSpeedDense(image_size=128)
net.to(device)

optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
print("Optimizer Initialized")

def dense_loss(output, gt_heatmap, gt_dense_data, is_free, alpha=0.9, decay=1):
    loss = 0
    batch_size = output.shape[0]
    w = gt_heatmap  # unit heatmap
    loss += (alpha * (1+w)* (output[:,:,:,0].unsqueeze(-1) - gt_heatmap)**2).sum()
    loss += ((1-alpha) * (1+w)* (output[:,:,:,1:] - gt_dense_data)**2).sum()
    return loss / batch_size

loss_fn = dense_loss
print("Loss function initialized")


device: cuda
Optimizer Initialized
Loss function initialized


In [4]:
from IPython.display import clear_output, display

def train_epoch_Centerspeed_dense(training_loader, net, optimizer, loss_fn,
                                  device='cpu', use_wandb=True):
    net.train()
    running_loss = 0.0
    total_batches = len(training_loader)
    start_epoch = time.time()
    last_step_time = time.time()

    # VSCode/노트북에서도 잘 동작하도록 auto tqdm 사용
    pbar = tqdm(training_loader, desc="Train", leave=False, dynamic_ncols=True)

    for i, batch in enumerate(pbar, 1):
        inputs, gts, data, dense_data, is_free = batch
        inputs     = inputs.to(device, dtype=torch.float32, non_blocking=True)
        gts        = gts.to(device, dtype=torch.float32, non_blocking=True)
        data       = data.to(device, dtype=torch.float32, non_blocking=True)
        dense_data = dense_data.to(device, dtype=torch.float32, non_blocking=True)
        is_free    = is_free.to(device, dtype=torch.float32, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        output = net(inputs)
        loss = loss_fn(output.permute(0, 2, 3, 1), gts.unsqueeze(-1), dense_data, is_free)

        loss.backward()
        optimizer.step()

        running_loss += float(loss.item())

        # ---- 실시간 ETA 표시 ----
        now = time.time()
        batch_time = now - last_step_time
        last_step_time = now
        # 남은 배치 × 평균 배치 시간 (간단히 현재 배치 시간 사용)
        remaining_batches = total_batches - i
        eta_sec = remaining_batches * batch_time
        pbar.set_postfix(loss=f"{loss.item():.4f}", eta=f"{eta_sec/60:.1f}m")

    avg_loss = running_loss / max(total_batches, 1)
    epoch_time = time.time() - start_epoch
    return avg_loss, epoch_time



@torch.no_grad()
def validate_epoch_Centerspeed_dense(validation_loader, net, loss_fn, device='cpu'):
    net.eval()
    running_loss = 0.0
    total = len(validation_loader)
    pbar = tqdm(validation_loader, desc="Val", leave=False, dynamic_ncols=True)

    for batch in pbar:
        inputs, gts, data, dense_data, is_free = batch
        inputs     = inputs.to(device, dtype=torch.float32, non_blocking=True)
        gts        = gts.to(device, dtype=torch.float32, non_blocking=True)
        data       = data.to(device, dtype=torch.float32, non_blocking=True)
        dense_data = dense_data.to(device, dtype=torch.float32, non_blocking=True)
        is_free    = is_free.to(device, dtype=torch.float32, non_blocking=True)

        output = net(inputs)
        loss = loss_fn(output.permute(0, 2, 3, 1), gts.unsqueeze(-1), dense_data, is_free)
        running_loss += float(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / max(total, 1)



In [5]:
EPOCHS = epochs
losses_train, losses_val, epoch_durations = [], [], []
start_all = time.time()

for epoch in range(EPOCHS):
    print(f"Epoch: {epoch}")

    train_loss, epoch_sec = train_epoch_Centerspeed_dense(
        training_loader=training_loader,
        net=net,
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=device,
        use_wandb=use_wandb
    )

    val_loss = validate_epoch_Centerspeed_dense(
        validation_loader=validation_loader,
        net=net,
        loss_fn=loss_fn,
        device=device
    )

    epoch_durations.append(epoch_sec)
    avg_epoch_sec = np.mean(epoch_durations)
    remaining_epochs = EPOCHS - (epoch + 1)
    eta_min = max(0.0, remaining_epochs * avg_epoch_sec) / 60.0

    losses_train.append(train_loss)
    losses_val.append(val_loss)

    # print(f'  train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | epoch_sec={epoch_sec:.2f}s | ETA={eta_min:.1f} min')

    if use_wandb and wandb.run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "time/epoch_sec": epoch_sec,
            "time/eta_minutes": eta_min
        }, commit=True)


Epoch: 0


Train:   0%|          | 0/1344 [00:00<?, ?it/s]

TypeError: 'bool' object is not callable